### Validation of Text Evaluation Metrics for Semantic Augmentation Tasks in Portuguese

#### Abstract

This document outlines a controlled experiment to assess and compare the suitability of a set of seven text evaluation metrics (Sacre-BLEU, ROUGE-L, METEOR, BERTScore, BLEURT, Perplexity, and COMET) for a specific task: measuring the semantic fidelity and intrinsic quality of Portuguese sentences subjected to various data augmentation techniques.

The objective is to identify which metrics:

1. Semantic Robustness: Maintain high scores for variations that preserve meaning (e.g., synonymy, paraphrase).
2. Semantic Sensitivity: Penalize scores for variations that degrade or factually alter the meaning.
3. Fluency Sensitivity: Identify and penalize agrammatical or non-fluent constructs.

#### Metrics Under Evaluation
1.  **SacreBLEU:** Traditional n-gram gold standard.
2.  **ROUGE-L:** Focuses on the Longest Common Subsequence (LCS).
3.  **METEOR:** N-gram with synonym matching and stemming.
4.  **BERTScore:** Cosine similarity between BERT embeddings (e.g., `neuralmind/bert-base-portuguese-cased`).
5.  **Perplexity (PPL):** Fluency measure calculated via a Causal LM (e.g., `neuralmind/gpt2-medium-portuguese`).
6.  **COMET:** State-of-the-art learned metric for translation (e.g., `wmt22-comet-da`).

In [1]:
import pandas as pd
import plotly.express as px
import evaluate
import polars as pl
import os

In [3]:
# N-GRAMS
sacrebleu = evaluate.load("sacrebleu")
rouge = evaluate.load('rouge')
meteor = evaluate.load('meteor')

# MODEL BASED
comet = evaluate.load("comet")
bertscore = evaluate.load('bertscore')

# PROBABILITY
perplexity = evaluate.load("perplexity")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/diegolopes/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/diegolopes/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/diegolopes/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/diegolopes/.pyenv/versions/3.10.19/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


# Section A: Controls (Sanity Checks)

#### **Group 1: Control (Identity Baseline)**
* **Purpose:** To validate metric functionality (sanity check).
* **Candidate (C1):** Identical to the Reference (R).
* **Hypothesized Outcome:** Similarity metrics (BLEU, ROUGE, METEOR, BERTScore, BLEURT, COMET) return a maximum score (~1.0). PPL returns the baseline fluency score (low/good).

In [2]:
original = [
    "Qual é a distância dos setores censitários até o limite da Região Norte, considerando apenas setores contidos no respectivo estado e a até 3 km do limite?"
]
augmented = [
    "Qual é a distância dos setores censitários até o limite da Região Norte, considerando apenas setores contidos no respectivo estado e a até 3 km do limite?"
]

sacrebleu_score = round(sacrebleu.compute(references=original, predictions=augmented)['score'], 2)
rouge_score = round(rouge.compute(references=original, predictions=augmented)['rougeL'], 2)
meteor_score = round(float(meteor.compute(references=original, predictions=augmented)['meteor']), 2)

#### **Grupo 2: Variação Trivial (Capitalização)**

- Propósito: Testar a robustez a mudanças superficiais.
- Candidato (C2): "qual é a distância dos setores censitários..." (Tudo em minúsculas).
- Hipótese de Resultado: Métricas modernas devem ser insensíveis (score ~1.0).

In [19]:
original = ["Quais trechos de setores censitários se sobrepõem à Região Nordeste e estão a até 2 km do limite regional?"]
augmented = ["censitários trechos do setores Quais se sobrepõem à Região Nordeste e estão a até 2 km de limite regional?"]


sacrebleu_score = round(sacrebleu.compute(references=original, predictions=augmented)['score'], 2)
rouge_score = round(rouge.compute(references=original, predictions=augmented)['rougeL'], 2)
bert_score = round(bertscore.compute(references=original, predictions=augmented, lang='pt', batch_size=8, device="cpu")['f1'][0], 2)

In [21]:
print(f"SacreBLEU: {sacrebleu_score/100}")
print(f"RougeL: {rouge_score}")
print(f"BertScore: {bert_score}")

SacreBLEU: 0.6375
RougeL: 0.77
BertScore: 0.89


In [2]:
path = "/Users/diegolopes/repositories/geo-nlq-to-sql/data"
df = pl.scan_parquet(f"{path}/geo_dataset", hive_partitioning=True).collect()

In [4]:
df.height

4900